In [56]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests

In [57]:
url = "https://ocw.mit.edu/ans7870/6/6.006/s08/lecturenotes/files/t8.shakespeare.txt"

text = requests.get(url).text

In [58]:
# Remove header
start = text.find("THE SONNETS")
if start == -1:
    start = 0

# Remove footer
end = text.find("End of Project Gutenberg")
if end == -1:
    end = len(text)

text = text[start:end]

print(text[:1000])  # preview clean text

THE SONNETS

by William Shakespeare



                     1
  From fairest creatures we desire increase,
  That thereby beauty's rose might never die,
  But as the riper should by time decease,
  His tender heir might bear his memory:
  But thou contracted to thine own bright eyes,
  Feed'st thy light's flame with self-substantial fuel,
  Making a famine where abundance lies,
  Thy self thy foe, to thy sweet self too cruel:
  Thou that art now the world's fresh ornament,
  And only herald to the gaudy spring,
  Within thine own bud buriest thy content,
  And tender churl mak'st waste in niggarding:
    Pity the world, or else this glutton be,
    To eat the world's due, by the grave and thee.


                     2
  When forty winters shall besiege thy brow,
  And dig deep trenches in thy beauty's field,
  Thy youth's proud livery so gazed on now,
  Will be a tattered weed of small worth held:  
  Then being asked, where all thy beauty lies,
  Where all the treasure of thy lusty d

In [59]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)

In [60]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [61]:
batch_size = 32
block_size = 128
embed_size = 256
num_heads = 8
num_layers = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix]).to(device)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]).to(device)

    return x, y

In [62]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(embed_size, head_size, bias=False)
        self.query = nn.Linear(embed_size, head_size, bias=False)
        self.value = nn.Linear(embed_size, head_size, bias=False)
        self.tril = torch.tril(torch.ones(block_size, block_size)).to(device)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) / (C ** 0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        v = self.value(x)
        out = wei @ v
        return out

In [63]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, embed_size)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

In [64]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_size, 4 * embed_size),
            nn.ReLU(),
            nn.Linear(4 * embed_size, embed_size),
        )

    def forward(self, x):
        return self.net(x)

In [65]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        head_size = embed_size // num_heads
        self.sa = MultiHeadAttention(num_heads, head_size)
        self.ffwd = FeedForward()
        self.ln1 = nn.LayerNorm(embed_size)
        self.ln2 = nn.LayerNorm(embed_size)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [66]:
class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_size)
        self.position_embedding = nn.Embedding(block_size, embed_size)

        self.blocks = nn.Sequential(*[Block() for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(embed_size)
        self.head = nn.Linear(embed_size, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb

        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)

        if targets is None:
            return logits, None

        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)

        loss = F.cross_entropy(logits, targets)
        return logits, loss

In [67]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = TransformerModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

for step in range(5000):  # increase steps for big dataset
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item()}")

Step 0, Loss: 4.515507698059082
Step 100, Loss: 2.563960313796997
Step 200, Loss: 2.4004101753234863
Step 300, Loss: 2.2953319549560547
Step 400, Loss: 2.2599570751190186
Step 500, Loss: 2.1051955223083496
Step 600, Loss: 2.0620956420898438
Step 700, Loss: 1.954940915107727
Step 800, Loss: 1.9306154251098633
Step 900, Loss: 1.885222315788269
Step 1000, Loss: 1.8748818635940552
Step 1100, Loss: 1.7479195594787598
Step 1200, Loss: 1.753408670425415
Step 1300, Loss: 1.7812623977661133
Step 1400, Loss: 1.7233800888061523
Step 1500, Loss: 1.7400566339492798
Step 1600, Loss: 1.653794527053833
Step 1700, Loss: 1.6926497220993042
Step 1800, Loss: 1.6075009107589722
Step 1900, Loss: 1.6340465545654297
Step 2000, Loss: 1.5956860780715942
Step 2100, Loss: 1.5493289232254028
Step 2200, Loss: 1.5551713705062866
Step 2300, Loss: 1.5463612079620361
Step 2400, Loss: 1.5551531314849854
Step 2500, Loss: 1.5266908407211304
Step 2600, Loss: 1.4802381992340088
Step 2700, Loss: 1.4583923816680908
Step 2800,

In [68]:
def generate(model, max_new_tokens=200):
    idx = torch.zeros((1,1), dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits/0.8, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, next_idx), dim=1)

    return decode(idx[0].tolist())

print(generate(model))


LENOMANA. Let him outs promises it the villain.
  GRUMIO. What hath consent playmen deathful boy; they are in behold
    of my proming show him honest to your dears:
    What are may she did becanned 
